# LEDGAR → LoRA fine-tune of **Llama-3.1-Nemotron-Nano-8B** (NVIDIA H100)

GPU copy, **preset for Nemotron-Nano-8B**. This is the Track-B deliverable: a clean, measured
LoRA fine-tune of an NVIDIA model on LEDGAR, with before/after macro-F1 and eval depth.

Structure is identical to the Mac learning notebook (`ledgar_lora/`) — only the model,
precision, and scale changed for the H100.

**What it does:** instruction-frame LEDGAR → score base model (BEFORE) → LoRA fine-tune →
score again (AFTER) → per-class + held-out test eval.

> First run the install cell. Needs a CUDA GPU (H100). `bf16` on; batch/data scaled up.

In [ ]:
# --- install (run once per fresh node) ---
# Install the CUDA build of torch FIRST (cluster tip from the team):
#   pip install torch --index-url https://download.pytorch.org/whl/cu124
!pip install -q "transformers>=4.44" "datasets>=2.19,<3" peft trl accelerate scikit-learn
print("deps installed")

## Part 1 — Settings & data

Preset for Nemotron-8B on an H100. To compare against an open model (the deck's model-vs-model
story), just change `MODEL_NAME` to e.g. `Qwen/Qwen2.5-7B-Instruct` or a Sarvam model and rerun.

In [ ]:
import torch, numpy as np, random, re
from datasets import load_dataset

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# === NVIDIA model (preset). Swap this ONE line to compare an open model. ===
MODEL_NAME = "nvidia/Llama-3.1-Nemotron-Nano-8B-v1"
# Open-model alternative for the head-to-head:
# MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

TRAIN_SIZE = 8000     # scale up on the GPU (up to 60000)
VAL_SIZE   = 1000
MAX_NEW    = 12       # tokens generated per answer (labels are short)

assert torch.cuda.is_available(), "This notebook expects a CUDA GPU (H100)."
device = "cuda"
print("GPU:", torch.cuda.get_device_name(0))

ds = load_dataset("coastalcph/lex_glue", "ledgar")
label_names = ds["train"].features["label"].names
train_raw = ds["train"].shuffle(seed=SEED).select(range(TRAIN_SIZE))
val_raw   = ds["validation"].shuffle(seed=SEED).select(range(VAL_SIZE))
print("labels:", len(label_names), "| train:", len(train_raw), "| val:", len(val_raw))

## Part 2 — Instruction framing

Generative models classify by *writing the label*. We frame each clause as a chat prompt with
the allowed label list, and (for training) append the correct label.

In [ ]:
LABELS_HINT = ", ".join(label_names)

def make_prompt(clause):
    return [
        {"role": "system", "content":
            "You are a legal clause classifier. Reply with ONLY the provision type, "
            "chosen from this list: " + LABELS_HINT},
        {"role": "user", "content": f"Clause:\n{clause}\n\nProvision type:"},
    ]

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("tokenizer ready. Example prompt:\n",
      tokenizer.apply_chat_template(make_prompt(train_raw[0]["text"]),
                                    add_generation_prompt=True, tokenize=False)[:400], "...")

## Part 3 — BEFORE: score the base Nemotron (parse text → label)

Batched generation on the GPU is fast. We parse each free-text answer back to one of the 100
labels, then compute macro-F1.

In [ ]:
from transformers import AutoModelForCausalLM
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.bfloat16, device_map="auto")
base_model.eval()

def parse_to_label(text):
    t = text.strip().lower()
    for lab in sorted(label_names, key=len, reverse=True):
        if lab.lower() in t:
            return lab
    tw = set(re.findall(r"\w+", t)); best, bs = label_names[0], -1
    for lab in label_names:
        s = len(tw & set(re.findall(r"\w+", lab.lower())))
        if s > bs: best, bs = lab, s
    return best

@torch.no_grad()
def predict_batch(model, clauses, bs=16):
    preds = []
    input_device = next(model.parameters()).device
    for i in range(0, len(clauses), bs):
        chunk = clauses[i:i+bs]
        prompts = [tokenizer.apply_chat_template(make_prompt(c), add_generation_prompt=True, tokenize=False)
                   for c in chunk]
        enc = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=1024).to(input_device)
        out = model.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False,
                             pad_token_id=tokenizer.pad_token_id)
        for j in range(len(chunk)):
            gen = tokenizer.decode(out[j][enc["input_ids"].shape[1]:], skip_special_tokens=True)
            preds.append(parse_to_label(gen))
    return preds

from sklearn.metrics import accuracy_score, f1_score
def evaluate(model, dset, tag):
    clauses = [r["text"] for r in dset]
    gold = [label_names[r["label"]] for r in dset]
    pred = predict_batch(model, clauses)
    acc = accuracy_score(gold, pred)
    f1  = f1_score(gold, pred, average="macro", zero_division=0, labels=sorted(set(gold)))
    print(f"[{tag}] accuracy={acc:.3f}  macro_f1={f1:.3f}")
    return {"accuracy": acc, "macro_f1": f1, "gold": gold, "pred": pred}

before = evaluate(base_model, val_raw, "BEFORE (base Nemotron)")

## Part 4 — LoRA fine-tune

LoRA on the 8B model = train ~0.1% of the weights. For a bigger model, adapting more attention
projections (`q,k,v,o`) usually helps.

In [ ]:
from peft import LoraConfig, get_peft_model

def to_text(row):
    p = tokenizer.apply_chat_template(make_prompt(row["text"]), add_generation_prompt=True, tokenize=False)
    return {"text": p + " " + label_names[row["label"]] + tokenizer.eos_token}
train_text = train_raw.map(to_text, remove_columns=train_raw.column_names)

lora_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj","k_proj","v_proj","o_proj"],   # more coverage for a big model
    task_type="CAUSAL_LM",
)
ft_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.bfloat16, device_map="auto")
ft_model = get_peft_model(ft_model, lora_cfg)
ft_model.enable_input_require_grads()
print("test")
ft_model.print_trainable_parameters()

In [ ]:
from trl import SFTTrainer, SFTConfig

sft_cfg = SFTConfig(
    output_dir="nemotron-ledgar-lora",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    logging_steps=25,
    bf16=True,
    report_to="none"
)
trainer = SFTTrainer(model=ft_model, args=sft_cfg,
                     train_dataset=train_text, processing_class=tokenizer)
trainer.train()
trainer.save_model("nemotron-ledgar-lora")   # saves the LoRA adapter
print("LoRA training complete; adapter saved to nemotron-ledgar-lora/")

## Part 5 — AFTER + compare

In [ ]:
ft_model.eval()
after = evaluate(ft_model, val_raw, "AFTER (LoRA Nemotron)")

def pct(x): return f"{100*x:5.1f}%"
print("\n"+"="*46)
print(f"{'Metric':<14}{'BEFORE':>12}{'AFTER':>12}")
print("-"*46)
print(f"{'Accuracy':<14}{pct(before['accuracy']):>12}{pct(after['accuracy']):>12}")
print(f"{'Macro-F1':<14}{pct(before['macro_f1']):>12}{pct(after['macro_f1']):>12}")
print("="*46)
print(f"Macro-F1 change: {pct(after['macro_f1']-before['macro_f1'])} points")

## Part 6 — Eval depth: per-class, qualitative, held-out test

In [ ]:
from sklearn.metrics import classification_report
present = sorted(set(after["gold"]))
print("AFTER — per-class (validation):")
print(classification_report(after["gold"], after["pred"], labels=present, zero_division=0, digits=3))

# qualitative: cases fine-tuning changed
shown = 0
for i, row in enumerate(val_raw):
    true, b, a = label_names[row["label"]], before["pred"][i], after["pred"][i]
    if b != a:
        print("CLAUSE :", row["text"][:120].replace("\n"," "), "...")
        print("  TRUE :", true, "| BEFORE :", b, "| AFTER :", a,
              (" <-- fixed" if (a==true and b!=true) else ""))
        shown += 1
    if shown >= 6: break

# held-out test (touch ONCE)
test_raw = ds["test"].shuffle(seed=SEED).select(range(VAL_SIZE))
after_test = evaluate(ft_model, test_raw, "TEST (held-out)")

## Model-vs-model (the head-to-head)

Run this whole notebook twice — once with the Nemotron `MODEL_NAME`, once with the open model
(Qwen/Sarvam) — and record both AFTER macro-F1 numbers in one table. That comparison is the
Track-B technical story: *does fine-tuning help NVIDIA's model as much as an open one, on legal
data?*

| Model | BEFORE macro-F1 | AFTER macro-F1 | Δ |
|---|---|---|---|
| Nemotron-Nano-8B | ... | ... | ... |
| Qwen2.5-7B (open) | ... | ... | ... |
